### Manual: 03-venv

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Python 3.8 venv via `uv`.

Kaggle's system interpreter is 3.13 and the pinned wheels are cp38-only, so the
whole pinned stack lives in an isolated uv-managed interpreter. Idempotent: if
the venv already reports 3.8 this cell does nothing.
"""

import json
import os
import shutil
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

log = os.path.join(sc.LOGS, "venv.log")
report = {"env": sc.ENV, "python": sc.PY}

already, version = sc.venv_ok()
if already:
    report["skipped"] = "venv already at Python " + version
    print(report["skipped"])
else:
    uv = shutil.which("uv")
    if uv is None:
        rc, _ = sc.sh(
            '"%s" -m pip install -q uv' % sys.executable,
            log=log,
        )
        uv = shutil.which("uv") or "uv"
        print("uv:", uv)
    report["uv"] = sc.capture("%s --version" % uv)

    # 3.8.20 is what uv's python-build-standalone channel provides for cp38.
    rc, _ = sc.sh("%s python install 3.8.20" % uv, log=log)
    report["uv_python_install_rc"] = rc
    rc, _ = sc.sh('%s venv --python 3.8.20 "%s"' % (uv, sc.ENV), log=log)
    report["uv_venv_rc"] = rc
    ok, version = sc.venv_ok()
    if not ok:
        # Fallback: let uv resolve any 3.8 it has.
        rc, _ = sc.sh('%s venv --python 3.8 "%s"' % (uv, sc.ENV), log=log)
        report["uv_venv_38_any_rc"] = rc
        ok, version = sc.venv_ok()
    report["venv_ok"] = ok
    report["python"] = version

ok, version = sc.venv_ok()
report["final_python"] = version
report["final_ok"] = bool(ok)
print(json.dumps(report, indent=2, sort_keys=True))
sc.save_json("venv.json", report)
sc.update(venv=report)
assert ok, "could not create a Python 3.8 venv: see " + log
print("VENV_OK " + version)
